# Volume Dry-Up Filter for Breakout Entry on SPY
## Strategy Brief
This strategy seeks to identify breakout opportunities in the SPY ETF by filtering for periods of low trading volume, known as volume dry-up. The assumption is that a breakout is more likely to occur after a period of low volume, as this indicates a potential accumulation phase. The strategy enters a long position when a breakout is detected following a volume dry-up. Historical backtesting suggests that this approach can capture significant price movements, though it may also lead to false signals during choppy market conditions.
## References
- (No external references)

In [ ]:
!pip install yfinance pandas numpy matplotlib scipy

## Phase 1 - Trading Context
In this phase, we define the parameters and constants that will be used throughout the strategy. These include the lookback period for volume dry-up and the threshold for detecting a breakout.

In [ ]:
LOOKBACK_PERIOD = 20
VOLUME_THRESHOLD = 0.5
BREAKOUT_THRESHOLD = 0.02

## Phase 2 - Data Exploration
We will download historical price and volume data for SPY from Yahoo Finance, calculate the moving average of volume, and plot it alongside the price to visualize periods of volume dry-up.

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Download SPY data
data = yf.download('SPY', start='2010-01-01')

# Calculate moving average of volume
volume_ma = data['Volume'].rolling(window=LOOKBACK_PERIOD).mean()

# Plot price and volume moving average
plt.figure(figsize=(14, 7))
plt.plot(data['Close'], label='SPY Close Price')
plt.plot(volume_ma, label=f'{LOOKBACK_PERIOD}-Day MA of Volume', alpha=0.5)
plt.title('SPY Price and Volume Moving Average')
plt.xlabel('Date')
plt.ylabel('Price / Volume')
plt.legend()
plt.show()

## Phase 3 - Strategy Engineering
We will create a signal series based on volume dry-up and breakout conditions. A position is taken when the signal indicates a breakout following a volume dry-up.

In [ ]:
# Calculate volume dry-up condition
volume_dry_up = data['Volume'] < (volume_ma * VOLUME_THRESHOLD)

# Calculate breakout condition
price_breakout = data['Close'] > data['Close'].shift(1) * (1 + BREAKOUT_THRESHOLD)

# Generate signal
signal = volume_dry_up & price_breakout

# Create positions series
positions = pd.Series(np.where(signal, 1, 0), index=data.index)

## Phase 4 - Coding & Backtesting
We will backtest the strategy by calculating daily returns based on the positions and plot the resulting equity curve.

In [ ]:
# Shift positions for backtesting
daily_returns = data['Close'].pct_change()
strategy_returns = positions.shift(1) * daily_returns

# Calculate equity curve
equity_curve = (1 + strategy_returns).cumprod()

# Plot equity curve
plt.figure(figsize=(14, 7))
plt.plot(equity_curve, label='Strategy Equity Curve')
plt.title('Strategy Equity Curve')
plt.xlabel('Date')
plt.ylabel('Equity')
plt.legend()
plt.show()

## Phase 5 - Performance Evaluation
We will evaluate the performance of the strategy using metrics such as CAGR, Sharpe ratio, Sortino ratio, Calmar ratio, and maximum drawdown, and compare it to a buy-and-hold strategy.

In [ ]:
def calculate_performance_metrics(returns):
    cagr = (equity_curve[-1] ** (252 / len(returns))) - 1
    sharpe_ratio = returns.mean() / returns.std() * np.sqrt(252)
    downside_returns = returns[returns < 0]
    sortino_ratio = returns.mean() / downside_returns.std() * np.sqrt(252)
    max_drawdown = (equity_curve / equity_curve.cummax() - 1).min()
    calmar_ratio = cagr / abs(max_drawdown)
    return cagr, sharpe_ratio, sortino_ratio, calmar_ratio, max_drawdown

strategy_metrics = calculate_performance_metrics(strategy_returns.dropna())
buy_and_hold_returns = daily_returns.copy()
buy_and_hold_metrics = calculate_performance_metrics(buy_and_hold_returns.dropna())

# Create comparison table
comparison_table = pd.DataFrame({
    'Metric': ['CAGR', 'Sharpe Ratio', 'Sortino Ratio', 'Calmar Ratio', 'Max Drawdown'],
    'Strategy': strategy_metrics,
    'Buy and Hold': buy_and_hold_metrics
})

print(comparison_table)

## Phase 6 - Deploy & Monitor
We will create a function to download the latest data, compute today's signal, and print the position to be taken.

In [ ]:
def get_latest_signal():
    latest_data = yf.download('SPY', period='60d')
    volume_ma_latest = latest_data['Volume'].rolling(window=LOOKBACK_PERIOD).mean()
    volume_dry_up_latest = latest_data['Volume'] < (volume_ma_latest * VOLUME_THRESHOLD)
    price_breakout_latest = latest_data['Close'] > latest_data['Close'].shift(1) * (1 + BREAKOUT_THRESHOLD)
    latest_signal = volume_dry_up_latest & price_breakout_latest
    latest_position = np.where(latest_signal.iloc[-1], 'Long', 'Flat')
    print(f'Today\'s Position: {latest_position}')

get_latest_signal()